# settlement_loader — exploratory tests

Run each cell in order. The loader writes to `data/warehouse.duckdb` (created automatically).

In [1]:
import sys
sys.path.insert(0, '..')  # make src/ importable from tests/

In [8]:
import re
from pathlib import Path
from src.bronze.settlement_loader import load
from src.db import get_connection
import duckdb

## 1 - First load

In [ ]:
paysettler_dir = Path('../docs/sample-data/paysettler')
csv_files = sorted(paysettler_dir.glob('settlement_*.csv'))

total_rows = 0
for csv_file in csv_files:
    reference_date = re.search(r"settlement_(\d{4}-\d{2}-\d{2})\.csv", csv_file.name).group(1)
    n = load(csv_file, reference_date=reference_date)
    total_rows += n
    print(f'{csv_file.name} -> reference_date={reference_date}, {n} rows')

print(f'\nTotal rows processed: {total_rows}')

## 2 - Running twice must not duplicate rows in the table.

In [15]:
# roda o mesmo loop de novo
total_rows_2 = 0
for csv_file in csv_files:
    reference_date = re.search(r"settlement_(\d{4}-\d{2}-\d{2})\.csv", csv_file.name).group(1)
    total_rows_2 += load(csv_file, reference_date=reference_date)

conn = get_connection()
total = conn.execute('SELECT COUNT(*) FROM raw_paysettler_settlements').fetchone()[0]
print(f'Total rows in table: {total}  (expected == {total_rows}, not doubled)')

Total rows in table: 954  (expected == 954, not doubled)


## 3 - Row counts and status breakdown

In [20]:
result = conn.execute("""
    SELECT
        status,
        COUNT(*)                       AS total_rows,
        SUM(amount)                    AS total_amount,
        MIN(settled_at::DATE)          AS earliest_date,
        MAX(settled_at::DATE)          AS latest_date
    FROM raw_paysettler_settlements
    GROUP BY status
    ORDER BY status
""").df()
total_all = result['total_rows'].sum()
reversed_pct = result.loc[result['status'] == 'REVERSED', 'total_rows'].sum() / total_all * 100

expected_pct = 2.0
tolerance = 1.0  # pontos percentuais de folga

if abs(reversed_pct - expected_pct) > tolerance:
    print(f'AVISO  REVERSED representa {reversed_pct:.2f}% do total (esperado ~{expected_pct}%, fora da tolerância de ±{tolerance}pp)')
else:
    print(f'OK     REVERSED representa {reversed_pct:.2f}% do total (dentro do esperado ~{expected_pct}%)')

result

OK     REVERSED representa 1.36% do total (dentro do esperado ~2.0%)


,status,total_rows,total_amount,earliest_date,latest_date
0,REVERSED,13,943021.0,2025-03-01,2025-03-06
1,SETTLED,941,17195759.0,2025-03-01,2025-03-18


## 4 - Amount normalization
The CSV ships dirty values like R$ 32.245,91. Confirm they were parsed correctly.

In [11]:
# All amounts must be positive
bad = conn.execute("""
    SELECT COUNT(*) FROM raw_paysettler_settlements WHERE amount <= 0
""").fetchone()[0]
print(f'Rows with amount <= 0: {bad}  (expected 0)')

# Amount distribution
conn.execute("""
    SELECT
        MIN(amount)  AS min_amount,
        MAX(amount)  AS max_amount,
        AVG(amount)  AS avg_amount
    FROM raw_paysettler_settlements
""").df()
     

Rows with amount <= 0: 0  (expected 0)


,min_amount,max_amount,avg_amount
0,289.0,367981.0,19013.396226


## 5 - Timestamp normalization
The CSV has two date formats (+00:00 and Z). Both must land as valid TIMESTAMPTZ.

In [12]:
null_dates = conn.execute("""
    SELECT COUNT(*) FROM raw_paysettler_settlements WHERE settled_at IS NULL
""").fetchone()[0]
print(f'NULL settled_at: {null_dates}  (expected 0)')

conn.execute("""
    SELECT
        settled_at::DATE AS date,
        COUNT(*)         AS transactions
    FROM raw_paysettler_settlements
    GROUP BY date
    ORDER BY date
""").df()

NULL settled_at: 0  (expected 0)


,date,transactions
0,2025-03-01,46
1,2025-03-02,151
2,2025-03-03,189
3,2025-03-04,180
4,2025-03-05,183
5,2025-03-06,141
6,2025-03-07,57
7,2025-03-09,1
8,2025-03-12,1
9,2025-03-13,2


## 6 - Duplicate transaction_ids in source CSV
The CSV itself has 510 rows but 10 duplicate transaction_ids — the PK resolves them to 500 unique rows.

In [ ]:
raw = duckdb.connect()
dups_df = raw.execute(f"""
    SELECT transaction_id, COUNT(*) AS occurrences
    FROM read_csv_auto({[str(f) for f in csv_files]!r}, header=true, union_by_name=true)
    GROUP BY transaction_id
    HAVING occurrences > 1
    ORDER BY occurrences DESC
""").df()

n_dups = len(dups_df)
expected_max = 20  # ajuste conforme a taxa de sujeira esperada (DIRTY_RATES) e o total de rows gerado

if n_dups == 0:
    print('AVISO  Nenhuma duplicata encontrada no CSV ')
elif n_dups > expected_max:
    print(f'AVISO  {n_dups} transaction_ids duplicados — acima do esperado (~{expected_max})')
else:
    print(f'OK     {n_dups} transaction_ids duplicados — dentro do esperado (~{expected_max})')

dups_df

AVISO  Nenhuma duplicata encontrada no CSV — confirme se isso é esperado (dirty=False?)


,transaction_id,occurrences


: 

## Sample

In [14]:

conn.execute("""
    SELECT * FROM raw_paysettler_settlements LIMIT 10
""").df()

,transaction_id,reference_date,merchant_id,amount,currency,settled_at,processor_reference,status,_loaded_at,_source_file
0,095b81d6-b97d-4617-b487-246c21f9c072,2025-03-01,MERCH_0008,2858.0,BRL,2025-03-01 22:47:32+01:00,PS-2025-0000000258,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
1,d6442365-1b27-45f6-b383-24f7eddcac0a,2025-03-01,MERCH_0006,66488.0,BRL,2025-03-01 18:21:04+01:00,PS-2025-0000000334,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
2,2d761ffa-7233-45af-9813-3da17d2120ad,2025-03-01,MERCH_0008,46740.0,BRL,2025-03-01 16:06:45+01:00,PS-2025-0000000160,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
3,0af5e4a9-b894-4df8-89ab-da915674b9e2,2025-03-01,MERCH_0007,17001.0,BRL,2025-03-01 07:20:55+01:00,PS-2025-0000000280,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
4,ab987d93-53bd-4128-a2f0-6403eac97d37,2025-03-01,MERCH_0008,10267.0,BRL,2025-03-01 21:07:27+01:00,PS-2025-0000000389,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
5,d202d65b-bdf1-4e53-a014-73df57686c78,2025-03-01,MERCH_0003,7158.0,BRL,2025-03-01 21:15:41+01:00,PS-2025-0000000875,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
6,5299e505-2976-4ab2-aa26-f7704ddb1b36,2025-03-01,MERCH_0004,2205.0,BRL,2025-03-01 11:31:14+01:00,PS-2025-0000000080,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
7,d590f435-a3fb-4d0b-b4be-28b44cd7e6f7,2025-03-01,MERCH_0008,5285.0,BRL,2025-03-01 19:05:20+01:00,PS-2025-0000000302,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
8,5a2615f8-9810-48c1-827d-eb45da6415b5,2025-03-01,MERCH_0008,5022.0,BRL,2025-03-01 22:12:57+01:00,PS-2025-0000000483,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv
9,04999aeb-d8b6-45b0-87c2-2f1b6d3026db,2025-03-01,MERCH_0001,5603.0,BRL,2025-03-01 10:14:31+01:00,PS-2025-0000000744,SETTLED,2026-07-15 14:09:13.763980,settlement_2025-03-01.csv


In [ ]:
conn.close()